# EDSS 전체 패널 빌드 검증

## TL;DR

전체 EDSS 재구성 결과가 233개 논리 패널, 265개 물리 단위, 278개 ZIP, 180,119,183행인지 확인한다. 카탈로그·프로필·실제 출력의 경로, 크기, SHA-256을 독립 재검산하고 기존 15개 패널의 재사용 안정성을 확인한다.

## 맥락과 검증 범위

원본 사전검사는 278개 ZIP의 1,142개 CSV를 읽어 행 폭 오류 없이 180,119,183행을 기록했다. 이 노트북은 빌드가 그 범위를 빠짐없이 233개 논리 패널로 만들었는지 검증한다. 행 내용의 후보키 grain, 개방ID 의미, 고아 키 재감사는 다음 실행 단위의 범위다.

In [1]:
import csv
import hashlib
import json
from collections import defaultdict
from pathlib import Path

repo = Path.cwd()
if not (repo / "data" / "metadata" / "edss_panel_catalog.csv").exists():
    repo = repo.parent
metadata = repo / "data" / "metadata"
processed = repo / "data" / "processed" / "edss"

with (metadata / "edss_panel_catalog.csv").open(encoding="utf-8-sig", newline="") as handle:
    catalog = list(csv.DictReader(handle))
quality = json.loads((metadata / "edss_panel_quality_report.json").read_text(encoding="utf-8"))
scan_summary = json.loads((metadata / "edss_full_rebuild_schema_scan_summary.json").read_text(encoding="utf-8"))
inventory_summary = json.loads((metadata / "edss_full_rebuild_inventory_summary.json").read_text(encoding="utf-8"))

len(catalog), quality["logical_dataset_count"], scan_summary["physical_unit_count"]

(233, 233, 265)

## 핵심 수량과 파일 완전성

In [2]:
expected_rows = 180_119_183
assert len(catalog) == 233
assert quality["logical_dataset_count"] == 233
assert quality["physical_dataset_count"] == 265
assert quality["raw_archive_count"] == 278
assert scan_summary["physical_unit_count"] == 265
assert scan_summary["archive_count"] == 278
assert sum(int(row["row_count"]) for row in catalog) == expected_rows
assert sum(int(profile["row_count"]) for profile in quality["profiles"]) == expected_rows

missing_outputs = [row["output_path"] for row in catalog if not (repo / row["output_path"]).exists()]
temporary_paths = list(processed.rglob("*.part")) + list(processed.rglob(".row-digests-*"))
assert missing_outputs == []
assert temporary_paths == []

total_bytes = sum(int(row["output_bytes"]) for row in catalog)
print({
    "logical_panels": len(catalog),
    "physical_units": quality["physical_dataset_count"],
    "raw_archives": quality["raw_archive_count"],
    "rows": expected_rows,
    "compressed_bytes": total_bytes,
    "missing_outputs": len(missing_outputs),
    "temporary_paths": len(temporary_paths),
})

{'logical_panels': 233, 'physical_units': 265, 'raw_archives': 278, 'rows': 180119183, 'compressed_bytes': 17000115337, 'missing_outputs': 0, 'temporary_paths': 0}


## 카탈로그와 프로필 대조

In [3]:
profiles = []
for path in processed.rglob("profile.json"):
    profile = json.loads(path.read_text(encoding="utf-8"))
    profile["_profile_path"] = path.relative_to(repo).as_posix()
    profiles.append(profile)

catalog_by_key = {(row["source"], row["catalog_code"], row["dataset"]): row for row in catalog}
profiles_by_key = {(p["source"], p["catalog_code"], p["dataset"]): p for p in profiles}
assert len(profiles) == 233
assert set(catalog_by_key) == set(profiles_by_key)

profile_mismatches = []
for key, row in catalog_by_key.items():
    profile = profiles_by_key[key]
    checks = {
        "row_count": int(row["row_count"]) == int(profile["row_count"]),
        "output_bytes": int(row["output_bytes"]) == int(profile["output_bytes"]),
        "output_sha256": row["output_sha256"] == profile["output_sha256"],
        "output_path": row["output_path"] == profile["output_path"],
    }
    if not all(checks.values()):
        profile_mismatches.append((key, checks))
assert profile_mismatches == []
print({"profile_files": len(profiles), "profile_mismatches": len(profile_mismatches)})

{'profile_files': 233, 'profile_mismatches': 0}


## 실제 출력 SHA-256 독립 재검산

In [4]:
checksum_failures = []
for index, row in enumerate(catalog, start=1):
    digest = hashlib.sha256()
    with (repo / row["output_path"]).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    actual = digest.hexdigest()
    if actual != row["output_sha256"]:
        checksum_failures.append((row["source"], row["catalog_code"], row["dataset"]))
assert checksum_failures == []
print({"checked_outputs": len(catalog), "checksum_failures": len(checksum_failures)})

{'checked_outputs': 233, 'checksum_failures': 0}


## 분야별 결과와 대형 패널

In [5]:
source_summary = defaultdict(lambda: {"panels": 0, "rows": 0, "compressed_bytes": 0})
for row in catalog:
    record = source_summary[row["source"]]
    record["panels"] += 1
    record["rows"] += int(row["row_count"])
    record["compressed_bytes"] += int(row["output_bytes"])

for source, record in sorted(source_summary.items()):
    print(source, record)

print("\n행 수 상위 10개 패널")
for row in sorted(catalog, key=lambda item: int(item["row_count"]), reverse=True)[:10]:
    print(row["source"], row["catalog_code"], row["dataset"], int(row["row_count"]), int(row["output_bytes"]))

고등교육통계 {'panels': 102, 'rows': 147761413, 'compressed_bytes': 13785638593}
대학정보공시 {'panels': 130, 'rows': 25032821, 'compressed_bytes': 2467051900}
취업통계 {'panels': 1, 'rows': 7324949, 'compressed_bytes': 747424844}

행 수 상위 10개 패널
고등교육통계 0230 지원자및입학현황 37847580 3529253771
고등교육통계 0240 학생동태현황_제적사유별 14643270 1325057374
고등교육통계 0209 연령별신입생현황_대학원 13729198 1327505020
고등교육통계 0211 연령별재적학생현황 10209800 919647319
고등교육통계 0214 연령별졸업생현황 9275040 832251476
고등교육통계 0228 졸업후상황 8979798 853544008
고등교육통계 0210 연령별입학현황 8506530 763675397
취업통계 0001 학생인적취업정보 7324949 747424844
고등교육통계 0241 학생동태현황_제적사유별_대학원 5776446 524671621
고등교육통계 0212 연령별재적학생현황_대학원 5245230 474902259


## 결론과 다음 단계

- 전체 233개 출력이 존재하고 카탈로그 SHA-256과 일치한다.
- 카탈로그와 233개 프로필의 경로·행 수·크기·체크섬 불일치는 0건이다.
- 행 수 합계는 원본 사전검사와 같은 180,119,183행이다.
- 취업통계 1개는 민감 가능 열 때문에 제한 경로에 유지된다.
- 다음 단계는 전체 출력에서 후보키 grain, 개방ID 결측·cardinality, 정확 중복, 학교연도 미연결을 독립 재계산하는 것이다.